# 02 — Limpieza y transformación (NuDat 3)

**Objetivo:** tipar, parsear modos, unificar $t_{1/2}$ y escribir `data/processed/` para MySQL.

**Pregunta:** ¿cómo se relaciona $N/Z$ con el modo dominante ($\beta^-$ vs $\beta^+$/EC) en estados base?

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
ROOT = Path("..").resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = Path(".").resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)
DOCS = ROOT / "docs"

## 0. Problemas detectados en la EDA

| Problema | Decisión |
|----------|----------|
| Unidades mixtas de $t_{1/2}$ | Convertir a s; `STABLE` → flag |
| Cobertura incompleta Chart HL | Preferir Chart en GS; si no, Wallet |
| Resonancias | `is_resonance` |
| Texto libre en Decay Modes | Parseo + modo dominante por branching |

## 1. Cargar raw

In [ ]:
wallet_raw = pd.read_csv(RAW / "walletcards.csv")
hl_raw = pd.read_csv(RAW / "nndc_nudat_data_export (10).csv")
print("wallet", wallet_raw.shape, "| hl", hl_raw.shape)

## 2. Transformar Wallet

In [ ]:
HALF_LIFE_TO_SECONDS = {
    "ys": 1e-24, "zs": 1e-21, "as": 1e-18, "fs": 1e-15, "ps": 1e-12,
    "ns": 1e-9, "us": 1e-6, "µs": 1e-6, "ms": 1e-3, "s": 1.0,
    "m": 60.0, "h": 3600.0, "d": 86400.0, "y": 365.25 * 86400.0,
}

def to_float(s):
    return pd.to_numeric(s, errors="coerce")

def wallet_hl_to_s(value, unit):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return np.nan
    text = str(value).strip()
    if not text or text.upper() == "STABLE":
        return np.nan
    try:
        number = float(text)
    except ValueError:
        return np.nan
    if unit is None or (isinstance(unit, float) and pd.isna(unit)):
        return np.nan
    factor = HALF_LIFE_TO_SECONDS.get(str(unit).strip().lower())
    return number * factor if factor is not None else np.nan

w = pd.DataFrame({
    "Z": to_float(wallet_raw["Atomic Number (Z)"]).astype("Int64"),
    "A": to_float(wallet_raw["Atomic Mass (A)"]).astype("Int64"),
    "level_index": to_float(wallet_raw["Level Index"]).astype("Int64"),
    "element": wallet_raw["Element"],
    "level_energy": to_float(wallet_raw["Level Energy"]),
    "level_energy_unit": wallet_raw["Level Energy (Unit)"],
    "spin_parity": wallet_raw["Spin-Parity"],
    "half_life_raw": wallet_raw["Half-Life"],
    "half_life_unit": wallet_raw["Half-Life (Unit)"],
    "abundance": to_float(wallet_raw["Abundance"]),
    "mass_excess_keV": to_float(wallet_raw["Mass Excess"]),
    "decay_modes_raw": wallet_raw["Decay Modes"],
    "decay_width": to_float(wallet_raw["Decay Width"]),
})
w["N"] = w["A"] - w["Z"]
unit = w["level_energy_unit"].astype(str).str.strip().str.lower()
scale = unit.map({"kev": 1.0, "mev": 1e3, "ev": 1e-3}).fillna(1.0)
w["level_energy_keV"] = w["level_energy"] * scale
w["is_stable"] = w["half_life_raw"].astype(str).str.upper().eq("STABLE")
w["is_resonance"] = w["decay_width"].notna() & w["half_life_raw"].isna() & ~w["is_stable"]
w["half_life_s_wallet"] = [wallet_hl_to_s(v, u) for v, u in zip(w["half_life_raw"], w["half_life_unit"])]
w.head()

## 3. Parsear modos

In [ ]:
def normalize_mode_token(token: str) -> str:
    t = token.strip().upper().replace("Α", "A").rstrip("?").strip()
    if t.startswith("B-"):
        if t.startswith(("B-N", "B-2N", "B-3N")):
            return "B-n"
        if t.startswith("B-A"):
            return "B-a"
        return "B-"
    if t.startswith("EC+B+") or t == "EC+B+":
        return "EC+B+"
    if t.startswith("EC"):
        return "ECp" if ("P" in t and t != "EC") else "EC"
    if t.startswith("B+"):
        return "B+"
    if t in {"A", "ALPHA"} or t.startswith("A=") or t == "A?":
        return "A"
    if t.startswith("IT"):
        return "IT"
    if t.startswith("N") and not t.startswith("NN"):
        return "n"
    if t.startswith("P"):
        return "p"
    if t.startswith("F"):
        return "SF"
    return t or "UNKNOWN"

def parse_decay_modes(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    text = str(raw).strip()
    if not text:
        return []
    channels = []
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        m = re.match(r"^(.+?)\s*(?:=|~)\s*([0-9]*\.?[0-9]+)\s*([0-9]*\.?[0-9]+)?\s*$", part)
        if m:
            channels.append({"mode_code": normalize_mode_token(m.group(1)), "branching_pct": float(m.group(2))})
        else:
            channels.append({"mode_code": normalize_mode_token(part), "branching_pct": None})
    return channels

def dominant_mode_class(channels, is_stable):
    if is_stable:
        return "STABLE"
    if not channels:
        return "UNKNOWN"
    ranked = sorted(channels, key=lambda c: (-1 if c["branching_pct"] is None else -c["branching_pct"]))
    code = ranked[0]["mode_code"]
    if code in {"B-", "B-n", "B-a"}:
        return "B-"
    if code in {"EC", "EC+B+", "B+", "ECp"}:
        return "EC_BP"
    if code == "A":
        return "ALPHA"
    if code == "IT":
        return "IT"
    return "OTHER"

w["dominant_mode"] = [dominant_mode_class(parse_decay_modes(r), bool(s)) for r, s in zip(w["decay_modes_raw"], w["is_stable"])]
w["dominant_mode"].value_counts()

## 4. Chart half-life: tipar y deduplicar

In [ ]:
hl = pd.DataFrame({
    "Z": to_float(hl_raw["z"]).astype("Int64"),
    "N": to_float(hl_raw["n"]).astype("Int64"),
    "half_life_s_chart": to_float(hl_raw["halflife(Seconds)"]),
}).drop_duplicates(["Z", "N"], keep="first")
print("hl:", len(hl))

## 5. Unificar $t_{1/2}$ → `nuclear_states`

In [ ]:
states = w.merge(hl, on=["Z", "N"], how="left")

def pick_half_life(row):
    if row["is_stable"]:
        return np.nan
    if row["level_index"] == 0 and pd.notna(row["half_life_s_chart"]):
        return row["half_life_s_chart"]
    return row["half_life_s_wallet"]

states["half_life_s"] = states.apply(pick_half_life, axis=1)
states["half_life_source"] = states.apply(
    lambda r: "none" if r["is_stable"] else (
        "chart" if r["level_index"] == 0 and pd.notna(r["half_life_s_chart"]) else (
            "wallet" if pd.notna(r["half_life_s_wallet"]) else "none"
        )
    ),
    axis=1,
)

nuclear_states = states[[
    "Z", "N", "A", "element", "level_index", "level_energy_keV", "spin_parity",
    "mass_excess_keV", "abundance", "half_life_s", "half_life_source",
    "is_stable", "is_resonance", "dominant_mode", "decay_modes_raw",
]].copy()
nuclear_states.head()

## 6. `nuclides.csv` (solo estados base)

In [ ]:
gs = nuclear_states[nuclear_states["level_index"] == 0].copy()
gs["name"] = gs["A"].astype(int).astype(str) + gs["element"].astype(str)
gs["N_over_Z"] = gs["N"] / gs["Z"].replace(0, np.nan)
nuclides = gs[[
    "Z", "N", "A", "element", "name", "spin_parity", "mass_excess_keV", "abundance",
    "half_life_s", "half_life_source", "is_stable", "is_resonance",
    "dominant_mode", "decay_modes_raw", "N_over_Z",
]].sort_values(["Z", "A"]).reset_index(drop=True)
nuclides.head()

## 7. `decay_channels`

In [ ]:
rows = []
for _, r in nuclear_states.iterrows():
    for ch in parse_decay_modes(r["decay_modes_raw"]):
        rows.append({
            "Z": int(r["Z"]), "A": int(r["A"]), "N": int(r["N"]),
            "level_index": int(r["level_index"]),
            "mode_code": ch["mode_code"], "branching_pct": ch["branching_pct"],
        })
decay_channels = pd.DataFrame(rows)
print(len(decay_channels))

## 8. Validación y escritura

In [ ]:
assert nuclides.duplicated(["Z", "A"]).sum() == 0
assert nuclear_states.duplicated(["Z", "A", "level_index"]).sum() == 0

nuclides.to_csv(PROCESSED / "nuclides.csv", index=False)
nuclear_states.to_csv(PROCESSED / "nuclear_states.csv", index=False)
decay_channels.to_csv(PROCESSED / "decay_channels.csv", index=False)

vc = nuclides["dominant_mode"].value_counts()
print(vc)
print("GS", len(nuclides), "states", len(nuclear_states), "channels", len(decay_channels))

log = [
    "# Log de limpieza NuDat", "",
    "Generado por `notebooks/02_clean_etl.ipynb`. Raw en `data/raw/` **no modificado**.", "",
    "## Pregunta que orienta el ETL", "",
    "¿Cómo se relaciona $N/Z$ con el modo de desintegración dominante ($\\beta^-$ vs $\\beta^+$/EC) en estados base?", "",
    "## Conteos", "",
    f"- nuclides (GS): {len(nuclides)}",
    f"- nuclear_states: {len(nuclear_states)}",
    f"- decay_channels: {len(decay_channels)}",
    f"- STABLE (GS): {int(nuclides['is_stable'].sum())}",
    f"- resonancias (GS): {int(nuclides['is_resonance'].sum())}", "",
    "## Decisiones clave", "",
    "- $t_{1/2}$ en segundos; Chart preferido en GS; `STABLE` → NULL + flag.",
    "- Modo dominante por mayor branching; clases B-, EC_BP, ALPHA, IT, OTHER, STABLE.",
    "- Chart half-life deduplicado por (Z,N).",
    "- Isómeros en `nuclear_states`; análisis N/Z–modo en GS (`level_index = 0`).",
    "- MySQL: 4 tablas (`element`, `nuclide`, `nuclear_state`, `decay_channel`).", "",
    "## Modos dominantes (GS)", "",
]
for mode, n in vc.items():
    log.append(f"- `{mode}`: {int(n)}")
(DOCS / "cleaning_log.md").write_text("\n".join(log) + "\n", encoding="utf-8")
print("written cleaning_log + processed")

## 9. Checklist

- [x] Raw no sobrescrito
- [x] `nuclides.csv`, `nuclear_states.csv`, `decay_channels.csv`
- [x] Listo para `python -m src.load_db`